# 05 · Metingen ontvangen en opslaan

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

In notebook 04 zag je berichten terugkomen zolang een ontvangende cel actief was. Om metingen te bewaren, moet een programma blijven luisteren en de inhoud naar een database schrijven. Je installeert nu zo'n ontvanger op je eigen DigitalOcean-server.

Je slaat eerst je eigen meetstroom op. Daarna voeg je een abonnement op de broker van je partner toe. Uiteindelijk bevat jouw serverdatabase zowel gewichten als voederopnames. Je controleert niet alleen de aantallen, maar ook of de ontvangen inhoud overeenkomt met de gepubliceerde brongegevens.

**Nodig:** je bestaande Codespace en eigen branch, de werkdatabase uit notebook 02, je eigen server uit notebook 03 en een geslaagde oefening 04. De docent wijst je een partner met de andere rol toe. Je werkt op je eigen server en gebruikt bij je partner alleen het MQTT-account **lezer**.

Voer de cellen in volgorde uit. Sommige stappen gebeuren in een SSH-terminal op je server, andere in het notebook. Bij een pauzekader rond je eerst de aangegeven serverhandeling af. De bewaarde uitvoer toont een voorbeeld. Tijdens je eigen uitvoering verschijnen jouw deelnemerscode en resultaten.

</div>

## 1. Je werkplek en gegevenslocaties controleren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 05)

Je oorspronkelijke werkdatabase blijft in Codespaces. De ontvanger maakt op de server een andere database, `verzamelde_metingen.db`. Die bevat de records die werkelijk via MQTT zijn aangekomen. Een databasebestand wordt dus niet tussen computers gedeeld. Beide computers hebben hun eigen bestand.

De cel hergebruikt de instellingen uit notebooks 01 en 03. Je hoeft geen nieuwe rol of server te kiezen.

</div>

In [1]:
# Tijdelijke hulpcel, verwijderd bij het genereren van de studentenversie.
_DOCENT_BRANCH_OVERRIDE = "student/proefdocent"

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
import hashlib
import ipaddress
import json
import re
import subprocess
import sys

r = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("Open notebook 05 vanuit de practicumrepository in Codespaces.")
hoofdmap = Path(r.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch) and "_DOCENT_BRANCH_OVERRIDE" in globals():
    branch = _DOCENT_BRANCH_OVERRIDE
instellingen = json.loads((hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json").read_text(encoding="utf-8"))
deelnemer, rol = instellingen["deelnemerscode"], instellingen["rol"]
if not re.fullmatch(r"[a-z0-9][a-z0-9_-]{0,62}", deelnemer) or branch != "student/" + deelnemer or instellingen.get("branch") != branch:
    raise ValueError("Controleer je eigen branch en de instellingen uit notebook 01.")
bestanden = {"gewichten": "asserva.db", "voederopname": "schauer.db"}
if rol not in bestanden or instellingen.get("bronbestand") != bestanden[rol]:
    raise ValueError("Rol en databank komen niet overeen.")
werkmap = hoofdmap / "gegevens/werk"
werkdatabase = werkmap / bestanden[rol]
server = json.loads((werkmap / "server.json").read_text(encoding="utf-8"))
if server.get("deelnemerscode") != deelnemer:
    raise ValueError("De serverinstellingen horen bij een andere deelnemer.")
serveradres = str(ipaddress.IPv4Address(server["ipv4"]))
ca = werkmap / "ca.crt"
for pad in [werkdatabase, ca]:
    if not pad.is_file(): raise FileNotFoundError(f"Ontbreekt: {pad.name}. Hervat notebook 02 of 03.")
sys.path[:0] = [str(hoofdmap / "programmatuur"), str(hoofdmap / "server")]
from publiceren import MqttSessie, lees_metingen, meettopic
from beginset_publiceren import publiceer_beginset
from meetgegevens import selectie_manifest
topic = meettopic(deelnemer, rol)
eigen_bron = "asserva" if rol == "gewichten" else "schauer"
print("Deelnemer:", deelnemer, "| Rol:", rol)
print("Werkdatabase:", werkdatabase.relative_to(hoofdmap))
print("Je eigen CA-vingerafdruk:", hashlib.sha256(ca.read_bytes()).hexdigest())
print("SSH:", f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

Deelnemer: alice | Rol: gewichten
Werkdatabase: gegevens/werk/asserva.db
Je eigen CA-vingerafdruk: 0eecabb5afca36cca7226b49e83147493085d19ff047f0469314e052d4a581b6
SSH: ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1


| Bestand of programma | Waar staat of draait het? | Functie |
|---|---|---|
| asserva.db of schauer.db | Codespace, gegevens/werk | Jouw aangevulde bronbestand |
| Publicatiecode | Codespace | Stuurt de gekozen bronrecords naar jouw broker |
| Mosquitto | Je eigen Droplet | Verdeelt jouw berichten |
| iot-ontvanger | Je eigen Droplet | Luistert naar jouw broker en de partnerbroker |
| verzamelde_metingen.db | Droplet, /var/lib/iot | Bewaart ontvangen bronrecords |
| Controlebestanden | Codespace, gegevens/inlevering | Bewaren aantallen, inhoudscontrole en antwoorden |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 1.** Waar moet de ontvanger draaien om te blijven werken als je de Codespace stopt?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 1):**

Voeg hier je antwoord in. Maak onderscheid tussen publiceren en ontvangen.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 1**

De ontvanger draait als service op de eigen Droplet. Hij blijft luisteren zolang die server, de dienst en de betrokken brokers actief zijn. De publicatie vanuit Codespaces stopt wanneer de Codespace stopt. Er worden dan geen nieuwe bronrecords vanuit die Codespace verstuurd.

</div>

## 2. De ontvangsoftware naar je eigen server kopiëren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Terminal, vanuit de hoofdmap van de repository)

De ontvangsoftware staat al in de repository. Je kopieert de bestanden naar je server, zodat ze daar onafhankelijk van Codespaces kunnen draaien. De Python-pakketten zijn in notebook 03 al in de virtuele omgeving geïnstalleerd.

Open een nieuwe terminal. Controleer met `pwd` dat je in de hoofdmap staat, waar ook `server/` staat. Voer onderstaande opdrachten uit. Vul het publieke IPv4-adres van je eigen Droplet in. Bij de wachtwoordvragen gebruik je je SSH-wachtwoord voor **beheer**. De bestanden komen eerst in een tijdelijke overdrachtsmap.

</div>

```bash
read -rp 'Publiek IPv4-adres van jouw Droplet: ' SERVERADRES
ssh -o HostKeyAlgorithms=ssh-ed25519 "beheer@$SERVERADRES" 'mkdir -p ~/practicum-ontvanger'
scp -o HostKeyAlgorithms=ssh-ed25519 server/meetgegevens.py server/ontvangen.py server/ontvangst_instellen.py server/ontvangst_status.py server/diensten/iot-ontvanger.service server/configuratie/ontvangst-capaciteit.conf "beheer@$SERVERADRES:~/practicum-ontvanger/"
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je eigen DigitalOcean-server (SSH-terminal, als beheer)

Open de SSH-verbinding met de opdracht uit de eerste codecel. Plaats de software onder `/opt/iot/programmatuur`. De servicegebruiker `iot` uit notebook 03 krijgt schrijfrechten op `/var/lib/iot`, waar zijn database staat. De programmabestanden blijven eigendom van root, zodat de ontvanger zijn eigen code niet kan aanpassen.

De opdrachten behouden bestaande meetgegevens. Als je hervat, mag je dezelfde software opnieuw installeren. Een al draaiende ontvanger gebruikt nieuwe code na een bewuste herstart.

</div>

```bash
sudo install -d -o root -g root -m 755 /opt/iot/programmatuur
sudo install -d -o iot -g iot -m 750 /var/lib/iot
sudo install -m 644 ~/practicum-ontvanger/meetgegevens.py ~/practicum-ontvanger/ontvangen.py ~/practicum-ontvanger/ontvangst_instellen.py ~/practicum-ontvanger/ontvangst_status.py /opt/iot/programmatuur/
sudo install -m 644 ~/practicum-ontvanger/iot-ontvanger.service /etc/systemd/system/iot-ontvanger.service
sudo install -m 644 ~/practicum-ontvanger/ontvangst-capaciteit.conf /etc/mosquitto/conf.d/ontvangst-capaciteit.conf
sudo systemctl daemon-reload
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Bij een volledige beginset kan de publisher sneller versturen dan een ontvanger opslaat. Mosquitto houdt dan tijdelijk berichten in een wachtrij. Het meegeleverde configuratiebestand begrenst die wachtrij per abonnee op 50.000 berichten en 64 MiB, en laat maximaal honderd onbevestigde uitgaande berichten tegelijk toe. Beide grenzen blijven eindig. Bij langdurige overbelasting kan ook deze wachtrij vollopen. Daarom blijft de inhoudscontrole na verzending noodzakelijk.

De nieuwe brokerinstellingen worden actief na de herstart in de volgende stap. Stem die herstart af met je partner als die al op jouw broker luistert.

</div>

## 3. Eerst je eigen stroom instellen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je eigen DigitalOcean-server (SSH-terminal, als beheer)

De ontvanger is een MQTT-client. Om te luisteren heeft hij het adres van de broker, een account en het CA-certificaat nodig. Voor luisteren volstaat het account **lezer**, ook op je eigen broker. Het account eigenaar blijft bedoeld voor publicatie vanuit je notebook.

De instelhulp vraagt je eigen deelnemerscode, rol, serveradres, CA-vingerafdruk en het MQTT-wachtwoord van **lezer**. Kopieer de vingerafdruk uit de eerste notebookcel. Hij moet overeenkomen met het bestand dat je in notebook 03 via de gecontroleerde SSH-verbinding hebt verkregen.

Het wachtwoord wordt verborgen ingevoerd. De ontvanger moet het later zelf kunnen gebruiken. Daarom bewaart de hulp het in `/etc/iot/ontvangst.geheim.json`, leesbaar voor root en de servicegroep iot. Het hoort uitsluitend op je server en niet in Git, screenshots of je notebookuitvoer.

</div>

```bash
sudo /opt/iot/venv/bin/python /opt/iot/programmatuur/ontvangst_instellen.py
```

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip: hervatten</strong><br>Bestaande instellingen worden behouden. Als je bewust een verkeerd adres of gewijzigd MQTT-wachtwoord wilt herstellen, voer je dezelfde opdracht uit met <code>--vervang</code>. Vul de gegevens opnieuw in en herstart daarna alleen de ontvanger. Laat het instellingenbestand nooit met <code>cat</code> afdrukken.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je eigen DigitalOcean-server (SSH-terminal, als beheer)

Start eerst de broker en daarna de ontvanger. `enable --now` laat de ontvanger nu starten én na een serverherstart terugkomen. Dit verandert de startinstelling van Mosquitto niet. De broker blijft handmatig te starten. Als de broker nog uitstaat, blijft de ontvanger proberen verbinding te maken.

Bekijk zowel de servicestatus als het logboek. `active (running)` betekent dat het programma draait. De regel **ABONNEMENT GEREED** met jouw code en rol bevestigt bovendien dat het abonnement is aanvaard.

</div>

```bash
sudo systemctl restart mosquitto
sudo systemctl enable --now iot-ontvanger
sudo systemctl status iot-ontvanger --no-pager
sudo journalctl -u iot-ontvanger -n 15 --no-pager
```

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle vóór publicatie</strong><br>Ga pas verder als <strong>ABONNEMENT GEREED</strong> verschijnt met jouw deelnemerscode en rol. Bij een fout gebruik je de herstelparagraaf onderaan. Een eerdere brokerbevestiging uit notebook 04 bewijst niet dat deze nieuwe ontvanger die metingen heeft opgeslagen.</div>

## 4. Twintig eigen records ontvangen en inspecteren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 05)

De ontvanger was tijdens notebook 04 nog niet actief. Publiceer daarom bewust opnieuw een selectie van twintig records. Deze cel gebruikt de publicatielijst uit notebook 04 niet om berichten over te slaan. Dezelfde bronrecords behouden dezelfde identiteit.

De publicatiecode wacht op brokerbevestigingen. Daarna controleer je apart de serverdatabase. Bij een eerste uitvoering verwacht je daar twintig unieke records van je eigen bron. Bij hervatten kunnen er al meer records staan.

</div>

In [3]:
selectie = list(lees_metingen(werkdatabase, rol, deelnemer, limiet=20))
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: ")) as sessie:
    proef_aantal = publiceer_beginset(sessie, topic, selectie)
assert proef_aantal == 20

Gereed: 20 publicaties bevestigd. Controleer nu de opslag op beide servers.


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je eigen DigitalOcean-server (SSH-terminal, als beheer)

SQLite bewaart de gegevens nu in één tabel `metingen`, waarin de kolom `bron` gewichten en voederopnames onderscheidt. Open de database alleen om te lezen. Deze twee opdrachten tonen de aantallen per bron en enkele identificaties. Noteer wat je werkelijk ziet.

</div>

```bash
sudo -u iot sqlite3 -readonly -header -column /var/lib/iot/verzamelde_metingen.db 'SELECT bron, COUNT(*) AS aantal FROM metingen GROUP BY bron;'
```

```bash
sudo -u iot sqlite3 -readonly -header -column /var/lib/iot/verzamelde_metingen.db 'SELECT bron, record_id, dier_id, hok, waarde, eenheid FROM metingen ORDER BY bron, record_id LIMIT 5;'
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** eerst op je DigitalOcean-server (SSH-terminal), daarna in Codespaces (Terminal)

Om het resultaat in je notebook te controleren, maak je een klein JSON-overzicht. Dit bevat aantallen, inhoudscontroles en verbindingsstatus, zonder wachtwoorden. Het is een momentopname van de database. Maak het na de publicatie opnieuw, want een eerder bestand wordt niet vanzelf bijgewerkt.

Voer eerst de volgende opdracht op de server uit. Ze toont het overzicht en bewaart dezelfde tekst in je beheermap.

</div>

```bash
sudo -u iot /opt/iot/venv/bin/python /opt/iot/programmatuur/ontvangst_status.py | tee ~/05_eigen_ontvangst.json
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Voer de volgende cel uit en kopieer de getoonde `scp`-opdracht naar een **Codespaces-terminal in de hoofdmap**, niet naar de SSH-terminal. Zo haal je het kleine controlebestand terug. Als de opslag nog bezig was, maak je het serveroverzicht opnieuw en herhaal je de download.

</div>

In [4]:
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}:~/05_eigen_ontvangst.json gegevens/werk/05_eigen_ontvangst.json")

scp -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1:~/05_eigen_ontvangst.json gegevens/werk/05_eigen_ontvangst.json


In [5]:
def lees_overzicht(pad):
    resultaat = json.loads(pad.read_text(encoding="utf-8"))
    gemaakt = datetime.fromisoformat(resultaat["gemaakt_op"])
    leeftijd = (datetime.now(timezone.utc) - gemaakt).total_seconds()
    if not -60 <= leeftijd <= 900:
        raise ValueError("Maak een nieuw serveroverzicht en download het opnieuw. Dit bestand is verouderd.")
    heartbeat = datetime.fromisoformat(resultaat["ontvanger_bijgewerkt_op"])
    if not 0 <= (gemaakt - heartbeat).total_seconds() <= 15:
        raise ValueError("De ontvanger werkte niet recent bij. Controleer de service en maak een nieuw overzicht.")
    return resultaat

eigen_overzicht = lees_overzicht(werkmap / "05_eigen_ontvangst.json")
if eigen_overzicht["bronnen"].get(eigen_bron, {}).get("aantal", 0) < len(selectie):
    raise ValueError("Er zijn nog onvoldoende eigen records opgeslagen. Controleer logboek en publicatie.")
print("Opgeslagen bronnen:", eigen_overzicht["bronnen"])
print("Verbindingen:", eigen_overzicht["verbindingen"])

Opgeslagen bronnen: {'asserva': {'aantal': 80028, 'sha256': '894618c5a3275a33c2fc75841814a4d18948351c70afc8cf6c1a65eccb7f0a58'}, 'schauer': {'aantal': 146649, 'sha256': '7d8c99caf64a6c128074b81dcf0de003d10f54887fe39a3489ebf0d52d85b249'}}
Verbindingen: [{'deelnemer': 'alice', 'rol': 'gewichten', 'toestand': 'gereed', 'nieuw': 80028, 'dubbel': 80, 'ongeldig': 1, 'conflict': 1, 'bijgewerkt_op': '2026-11-18T14:44:06.703527+00:00'}, {'deelnemer': 'bob', 'rol': 'voederopname', 'toestand': 'gereed', 'nieuw': 146649, 'dubbel': 40, 'ongeldig': 0, 'conflict': 0, 'bijgewerkt_op': '2026-11-18T14:44:03.672008+00:00'}]


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 2.** Hoeveel eigen records staan er op je server? Zoek de betekenis van bron, record_id en eerste_deelnemer op in de tabelstructuur.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 2):**

Voeg hier je antwoord in. Gebruik .schema metingen in SQLite of PRAGMA table_info(metingen). Noteer je eigen aantal en één record_id.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 2**

Bij de eerste uitvoering staan twintig eigen records in de serverdatabase. bron onderscheidt asserva en schauer. record_id is de ongewijzigde ROW_ID uit het bronbestand. Samen vormen ze de primaire sleutel. eerste_deelnemer vermeldt via welke publisher het record voor het eerst is opgeslagen en hoort niet bij de meetidentiteit. Bij hervatten kan het aantal hoger zijn.

</div>

## 5. De gegevens van je partner verzamelen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces en samen met je partner

Elke ontvanger kan twee clientverbindingen onderhouden: één met de eigen broker en één met de partnerbroker. De brokers hoeven niet met elkaar verbonden te worden. Jouw ontvanger krijgt leesrechten op de afgesproken partnerstroom.

Wissel de volgende gegevens uit: deelnemerscode, rol, publiek IPv4-adres, het openbare bestand `ca.crt` en de bijbehorende SHA-256-vingerafdruk. Je partner geeft daarnaast het MQTT-wachtwoord van **lezer** via een afgesproken besloten kanaal of rechtstreeks ter plaatse. Je deelt geen SSH-, eigenaar- of API-wachtwoord en geen privésleutel.

Deel ook jouw eigen openbare CA met je partner. Download die in Codespaces uit `gegevens/werk/ca.crt`. Je eigen vingerafdruk staat in de eerste notebookcel. Vergelijk de vingerafdruk met je partner via een vertrouwde route. Alleen een ontvangen bestand opnieuw hashen bewijst nog niet wie het heeft aangeleverd.

</div>

In [6]:
partnerpad = werkmap / "partner.json"
if partnerpad.exists():
    partner = json.loads(partnerpad.read_text(encoding="utf-8"))
else:
    partner = {
        "deelnemerscode": input("Deelnemerscode van je partner: ").strip(),
        "rol": "voederopname" if rol == "gewichten" else "gewichten",
        "ipv4": str(ipaddress.IPv4Address(input("Publiek IPv4-adres van je partner: ").strip())),
    }
    partnerpad.write_text(json.dumps(partner, indent=2) + "\n", encoding="utf-8")
if not re.fullmatch(r"[a-z0-9][a-z0-9_-]{0,62}", partner["deelnemerscode"]) or partner["deelnemerscode"] == deelnemer:
    raise ValueError("Controleer de deelnemerscode van je partner.")
if partner["rol"] != ("voederopname" if rol == "gewichten" else "gewichten"):
    raise ValueError("De partner moet de andere rol hebben.")
print("Partner:", partner["deelnemerscode"], "| Rol:", partner["rol"])

Partner: bob | Rol: voederopname


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Upload het ontvangen openbare certificaat naar `gegevens/werk/` in je eigen Codespace en geef het de naam **ca_partner.crt**. Voer dan de volgende cel uit en plak de vingerafdruk die je partner heeft bevestigd. De cel controleert de gehele bestandsinhoud. Een verschil los je eerst met je partner op.

</div>

In [7]:
partner_ca = werkmap / "ca_partner.crt"
bevestigde_vingerafdruk = input("Door je partner bevestigde SHA-256 van diens ca.crt: ").strip().lower()
if hashlib.sha256(partner_ca.read_bytes()).hexdigest() != bevestigde_vingerafdruk:
    raise ValueError("De vingerafdruk verschilt. Controleer het bestand met je partner.")
print("Het ontvangen CA-bestand komt overeen met de bevestigde vingerafdruk.")
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 gegevens/werk/ca_partner.crt beheer@{serveradres}:~/ca_partner.crt")

Het ontvangen CA-bestand komt overeen met de bevestigde vingerafdruk.
scp -o HostKeyAlgorithms=ssh-ed25519 gegevens/werk/ca_partner.crt beheer@127.0.0.1:~/ca_partner.crt


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Terminal), daarna op je eigen DigitalOcean-server (SSH-terminal)

Voer de getoonde kopieeropdracht in de Codespaces-terminal uit. Plaats vervolgens het openbare partnercertificaat op je eigen server en voeg de partnerbron toe. De instelhulp vraagt de partnergegevens en het wachtwoord van **lezer op de partnerbroker**. Plak dezelfde bevestigde vingerafdruk. Bij hervatten blijven bestaande partnerinstellingen behouden. Gebruik `--partner --vervang` alleen als je ze bewust moet wijzigen.

</div>

```bash
sudo install -o root -g iot -m 640 ~/ca_partner.crt /etc/iot/ca_partner.crt
sudo /opt/iot/venv/bin/python /opt/iot/programmatuur/ontvangst_instellen.py --partner
sudo systemctl restart iot-ontvanger
sudo journalctl -u iot-ontvanger -n 20 --no-pager
```

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle met je partner</strong><br>Controleer op <strong>beide servers</strong> twee regels <strong>ABONNEMENT GEREED</strong>: één voor gewichten en één voor voederopname, met de afgesproken codes. Begin pas daarna de volledige publicatie. Je partner doorloopt dezelfde stappen op zijn of haar eigen server. Laat beide brokers en ontvangers aan.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 3.** Met hoeveel brokers maakt jouw ontvanger nu verbinding en waarom is het lezerswachtwoord voldoende?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 3):**

Voeg hier je antwoord in. Benoem ook welke wachtwoorden je juist niet met je partner deelt.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 3**

De ontvanger maakt verbinding met de eigen broker en de partnerbroker. Op beide vraagt hij een abonnement aan. Hij publiceert geen metingen en heeft daarom alleen leesrechten nodig. De eigen publisher gebruikt het eigenaarsaccount op de eigen broker. Er is geen brokerbridge nodig.

</div>

## 6. De volledige beginsets uitwisselen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 05), door beide partners

De twintig proefrecords zijn slechts een klein deel van de eerste vier weken. Nu verstuurt iedere partner de volledige eigen beginset. In jouw werkbestand verwacht je 80.028 gewichtsrecords of 146.649 voederrecords. Beide serverdatabases moeten uiteindelijk beide bronnen bevatten.

Maak eerst een **broncontrolebestand**. Dat bevat het aantal records en een inhoudsvingerafdruk per bron. De vingerafdruk wordt berekend over de stabiele identificaties en de inhoud van alle berichten. Daardoor ontdek je ook een verschil wanneer het aantal records toevallig gelijk is. De deelnemerscode telt niet mee in de meetinhoud.

Een eigen hoktabel is onderdeel van je berichten. Je vergelijkt daarom met de controle van de partner die daadwerkelijk publiceert, niet met een veronderstelde voorbeeldwaarde. De code doet geen kwaliteitsselectie. Ook nullen en kwaliteitsvlaggen uit de bron blijven behouden.

</div>

In [8]:
eigen_manifest = selectie_manifest(lees_metingen(werkdatabase, rol, deelnemer, limiet=None))
verwacht_aantal = 80028 if rol == "gewichten" else 146649
if eigen_manifest[eigen_bron]["aantal"] != verwacht_aantal:
    raise ValueError("Gebruik de werkdatabase met de eerste vier weken. Controleer afwijkende aantallen eerst.")
broncontrole = {"deelnemerscode": deelnemer, "rol": rol, "bronnen": eigen_manifest}
broncontrolepad = hoofdmap / "gegevens/inlevering/05_broncontrole.json"
broncontrolepad.write_text(json.dumps(broncontrole, indent=2) + "\n", encoding="utf-8")
print(json.dumps(broncontrole, indent=2))

{
  "deelnemerscode": "alice",
  "rol": "gewichten",
  "bronnen": {
    "asserva": {
      "aantal": 80028,
      "sha256": "894618c5a3275a33c2fc75841814a4d18948351c70afc8cf6c1a65eccb7f0a58"
    }
  }
}


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Download `gegevens/inlevering/05_broncontrole.json` en bezorg het aan je partner. Upload het bestand van je partner naar je eigen Codespace onder **gegevens/werk/partner_broncontrole.json**. Het bevat geen wachtwoorden. Bewaar het gescheiden van je eigen broncontrole.

Voer de volgende cel pas uit wanneer beide ontvangers hun abonnementen hebben bevestigd. Je mag de publicatiecel na een onderbreking opnieuw uitvoeren. Alle bronrecords krijgen dezelfde identiteit. De ontvangers herkennen eerder opgeslagen records. Laat de cel uitlopen en lees de voortgang. Een brokerbevestiging is nog steeds geen bewijs van volledige databaseopslag.

</div>

In [9]:
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: "), timeout=30) as sessie:
    volledig_verstuurd = publiceer_beginset(
        sessie, topic, lees_metingen(werkdatabase, rol, deelnemer, limiet=None)
    )
assert volledig_verstuurd == verwacht_aantal

5000 publicaties door de broker bevestigd


10000 publicaties door de broker bevestigd


15000 publicaties door de broker bevestigd


20000 publicaties door de broker bevestigd


25000 publicaties door de broker bevestigd


30000 publicaties door de broker bevestigd


... (6 outputs left out)

65000 publicaties door de broker bevestigd


70000 publicaties door de broker bevestigd


75000 publicaties door de broker bevestigd


80000 publicaties door de broker bevestigd
Gereed: 80028 publicaties bevestigd. Controleer nu de opslag op beide servers.


## 7. De ontvangen inhoud controleren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je eigen DigitalOcean-server (SSH-terminal), daarna in Codespaces

Wacht tot beide partners hun beginset hebben gepubliceerd. De ontvanger kan de laatste berichten nog aan het verwerken zijn. Maak op je eigen server een nieuw overzicht. Bij twee volledige bronnen verwacht je samen **226.677 unieke records**.

De statusopdracht leest de aantallen en vingerafdrukken binnen één databaseleestransactie. Zo vergelijkt ze een samenhangende momentopname, terwijl de ontvanger kan blijven werken.

</div>

```bash
sudo -u iot /opt/iot/venv/bin/python /opt/iot/programmatuur/ontvangst_status.py | tee ~/05_ontvangstcontrole.json
```

In [10]:
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}:~/05_ontvangstcontrole.json gegevens/werk/05_ontvangstcontrole.json")

scp -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1:~/05_ontvangstcontrole.json gegevens/werk/05_ontvangstcontrole.json


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Voer de getoonde opdracht in de Codespaces-terminal uit. Controleer ook of `partner_broncontrole.json` is geüpload. Voer dan de volgende cel uit. Een melding over ontbrekende of afwijkende inhoud is een reden om ontvangst en herpublicatie te controleren, niet om records uit een bron te verwijderen.

</div>

In [11]:
partnercontrole = json.loads((werkmap / "partner_broncontrole.json").read_text(encoding="utf-8"))
partner_bron = "schauer" if rol == "gewichten" else "asserva"
if partnercontrole.get("deelnemerscode") != partner["deelnemerscode"] or partnercontrole.get("rol") != partner["rol"]:
    raise ValueError("Het broncontrolebestand hoort niet bij de afgesproken partner.")
if set(partnercontrole["bronnen"]) != {partner_bron}:
    raise ValueError("De partnercontrole bevat niet de verwachte bron.")
verwachte_bronnen = eigen_manifest | partnercontrole["bronnen"]
ontvangst = lees_overzicht(werkmap / "05_ontvangstcontrole.json")
if ontvangst["bronnen"] != verwachte_bronnen:
    raise ValueError("Aantallen of inhoud verschillen. Controleer beide ontvangers, herpubliceer ontbrekende bronnen en download een nieuw overzicht.")
actief = {(v["deelnemer"], v["rol"]) for v in ontvangst["verbindingen"] if v["toestand"] == "gereed"}
if actief != {(deelnemer, rol), (partner["deelnemerscode"], partner["rol"])}:
    raise ValueError("Niet beide afgesproken abonnementen zijn gereed. Controleer het logboek.")
print("Beide bronnen zijn volledig ontvangen en inhoudelijk gelijk aan de broncontroles.")
for bron, info in ontvangst["bronnen"].items():
    print(bron, info["aantal"], "unieke records")
print("Totaal:", sum(v["aantal"] for v in ontvangst["bronnen"].values()))

Beide bronnen zijn volledig ontvangen en inhoudelijk gelijk aan de broncontroles.
asserva 80028 unieke records
schauer 146649 unieke records
Totaal: 226677


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 4.** Waarom controleren we behalve aantallen ook de inhoudsvingerafdruk?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 4):**

Voeg hier je antwoord in. Geef één voorbeeld waarbij het aantal klopt maar de inhoud verschilt.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 4**

Een ontbrekende rij en een onterecht toegevoegde rij kunnen elkaar in een telling compenseren. Een gewijzigde meetwaarde verandert het aantal evenmin. De vingerafdruk vergelijkt identificaties en inhoud. Ze bewijst overeenkomst met het controlebestand, niet dat alle oorspronkelijke metingen wetenschappelijk bruikbaar zijn.

</div>

## 8. Dubbelen en ongeldige berichten onderzoeken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 05)

Een MQTT-bericht kan opnieuw aankomen, bijvoorbeeld door herpublicatie of verbindingsherstel. De primaire sleutel `(bron, record_id)` voorkomt dat dezelfde bronmeting een extra rij wordt. Identieke inhoud verhoogt alleen de teller `dubbel`. Afwijkende inhoud bij dezelfde identiteit wordt als `conflict` gemeld en overschrijft de bestaande meting niet.

Een bericht dat niet aan het afgesproken schema voldoet wordt als `ongeldig` geteld. De ontvanger blijft luisteren. In deze proef verstuur je eerst opnieuw twintig bestaande records en daarna één bewust onvolledig JSON-object. Dat object is een technische foutproef, geen verzonnen meting.

</div>

In [12]:
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: ")) as sessie:
    publiceer_beginset(sessie, topic, selectie)
    sessie.verstuur(topic, {"schema_versie": 999})
print("Twintig bestaande records en één bewust ongeldig bericht aangeboden.")

Gereed: 20 publicaties bevestigd. Controleer nu de opslag op beide servers.
Twintig bestaande records en één bewust ongeldig bericht aangeboden.


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal)

Bekijk het logboek. Je verwacht een melding **Bericht ongeldig**. Maak vervolgens een nieuw overzicht. De aantallen unieke metingen en hun inhoud moeten gelijk blijven, terwijl de tellers voor dubbelen en ongeldige berichten toenemen. De tellers lopen sinds de eerste inrichting door, ook over een serviceherstart heen.

</div>

```bash
sudo journalctl -u iot-ontvanger -n 15 --no-pager
sudo -u iot /opt/iot/venv/bin/python /opt/iot/programmatuur/ontvangst_status.py | tee ~/05_na_herhaling.json
```

In [13]:
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}:~/05_na_herhaling.json gegevens/werk/05_na_herhaling.json")

scp -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1:~/05_na_herhaling.json gegevens/werk/05_na_herhaling.json


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Voer de getoonde downloadopdracht uit in de Codespaces-terminal. Controleer daarna de vergelijking. Als het overzicht te vroeg is gemaakt, maak je het opnieuw zodra het logboek de afwijzing toont.

</div>

In [14]:
na_herhaling = lees_overzicht(werkmap / "05_na_herhaling.json")
assert na_herhaling["bronnen"] == ontvangst["bronnen"], "De unieke meetinhoud is onverwacht veranderd."
def eigen_tellers(overzicht):
    return next(v for v in overzicht["verbindingen"] if v["deelnemer"] == deelnemer and v["rol"] == rol)
voor, na = eigen_tellers(ontvangst), eigen_tellers(na_herhaling)
assert na["dubbel"] >= voor["dubbel"] + 20, "De herpublicaties zijn nog niet verwerkt."
assert na["ongeldig"] >= voor["ongeldig"] + 1, "De foutproef is nog niet verwerkt."
print("Unieke meetinhoud ongewijzigd.")
print("Extra dubbele berichten:", na["dubbel"] - voor["dubbel"])
print("Extra ongeldige berichten:", na["ongeldig"] - voor["ongeldig"])

Unieke meetinhoud ongewijzigd.
Extra dubbele berichten: 20
Extra ongeldige berichten: 1


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 5.** Welke tellers veranderden bij jouw proef en waarom veranderde het aantal metingen niet?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 5):**

Voeg hier je antwoord in. Noteer de verschillen uit jouw eigen uitvoer. Andere actieve publicaties kunnen de tellers extra laten stijgen.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 5**

De teller dubbel stijgt met minstens twintig en ongeldig met minstens één. De metingen waren al aanwezig met dezelfde bronidentiteit en inhoud. Het onvolledige object voldoet niet aan het schema en wordt afgewezen. Geen van deze berichten vormt een nieuwe geldige bronrij.

</div>

## 9. De dienst beheren en verbindingen herstellen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je eigen DigitalOcean-server (SSH-terminal, als beheer)

Een service blijft actief wanneer je de SSH-terminal sluit. Systemd bewaakt het programma en start het opnieuw bij een procesfout. De ontvanger probeert bij een verbroken MQTT-verbinding opnieuw verbinding te maken en vraagt daarna opnieuw een abonnement aan.

**Doe-opdracht:** spreek met je partner af dat jullie even geen nieuwe meetrecords publiceren. Herstart alleen de ontvanger. Controleer opnieuw twee regels ABONNEMENT GEREED. De opgeslagen metingen blijven bestaan. Een herstart maakt geen lege database.

</div>

```bash
sudo systemctl restart iot-ontvanger
sudo systemctl status iot-ontvanger --no-pager
sudo journalctl -u iot-ontvanger -n 20 --no-pager
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Stop en start vervolgens je eigen broker. De ontvanger op de partnerserver verliest daardoor tijdelijk ook zijn verbinding met jouw broker. Controleer samen dat beide ontvangers daarna opnieuw abonneren. Tijdens de onderbreking gemiste historie wordt niet automatisch teruggehaald. Herpubliceer na herstel de beginset wanneer er twijfel is over volledigheid.

</div>

```bash
sudo systemctl stop mosquitto
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Bekijk met je partner het logboek of een nieuw statusoverzicht. De verbinding met jouw broker hoort verbroken of onbereikbaar te worden. Start de broker vervolgens weer handmatig.

</div>

```bash
sudo systemctl start mosquitto
sudo journalctl -u iot-ontvanger -n 20 --no-pager
```

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Uitleg: na een volledige serverherstart</strong><br>De ontvanger is ingeschakeld en komt automatisch terug. Mosquitto blijft uit tot je <code>sudo systemctl start mosquitto</code> uitvoert. De ontvanger kan dus actief zijn terwijl zijn eigen broker nog niet bereikbaar is. Controleer daarna de abonnementen en herpubliceer zo nodig. Alleen de ontvangerservice wordt hier automatisch gestart, niet een installatiescript.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Maak na de herstartproef een nieuw overzicht en download het met de volgende opdracht. De laatste controle vergelijkt de inhoud met de volledige beginsets en controleert of beide abonnementen opnieuw gereed zijn.

</div>

```bash
sudo -u iot /opt/iot/venv/bin/python /opt/iot/programmatuur/ontvangst_status.py | tee ~/05_na_herstel.json
```

In [15]:
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}:~/05_na_herstel.json gegevens/werk/05_na_herstel.json")

scp -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1:~/05_na_herstel.json gegevens/werk/05_na_herstel.json


In [16]:
na_herstel = lees_overzicht(werkmap / "05_na_herstel.json")
assert na_herstel["bronnen"] == verwachte_bronnen
gereed = {(v["deelnemer"], v["rol"]) for v in na_herstel["verbindingen"] if v["toestand"] == "gereed"}
assert gereed == {(deelnemer, rol), (partner["deelnemerscode"], partner["rol"])}
print("Meetinhoud behouden en beide abonnementen opnieuw gereed.")

Meetinhoud behouden en beide abonnementen opnieuw gereed.


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 6.** Waarom kan active (running) samengaan met een ontbrekende partnerstroom? Wat controleer je na een serverherstart?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 6):**

Voeg hier je antwoord in. Beschrijf wat je zag toen je eigen broker gestopt was en wat je partner toen zag.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 6**

Active betekent dat het ontvangprogramma draait. De partnerbroker kan toch uitstaan, onbereikbaar zijn of de aanmelding weigeren. Na een serverherstart start je de eigen broker handmatig en controleer je de afzonderlijke abonnementen, logmeldingen en inhoudscontrole. Voor gemiste historie herpubliceer je na herstel.

</div>

## 10. Resultaat bewaren en afsluiten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 05, daarna Source Control)

Je bewaart een compact verslag van de broncontroles, de ontvangen inhoud en de foutproef. De verzameldatabase blijft op de server beschikbaar voor de volgende oefening. Kopieer een actief SQLite-bestand niet zomaar terwijl de ontvanger schrijft. De kleine overzichten worden via een leestransactie gemaakt.

Voer de volgende cel uit nadat alle controles geslaagd zijn. Je eigen antwoorden blijven in het notebook.

</div>

In [17]:
verslag = {
    "deelnemerscode": deelnemer, "branch": branch, "rol": rol,
    "partner": partner["deelnemerscode"], "getest_op_utc": datetime.now(timezone.utc).isoformat(),
    "bronnen": na_herstel["bronnen"], "volledigheid": "geslaagd",
    "herhaling_zonder_extra_metingen": na_herhaling["bronnen"] == ontvangst["bronnen"],
    "ongeldig_bericht_afgewezen": na["ongeldig"] > voor["ongeldig"],
    "abonnementen_na_herstel": "gereed",
}
inleverpad = hoofdmap / "gegevens/inlevering/05_ontvangstcontrole.json"
inleverpad.write_text(json.dumps(verslag, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Bewaard:", inleverpad.relative_to(hoofdmap))

Bewaard: gegevens/inlevering/05_ontvangstcontrole.json


<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Let op: bewaar je werk en stop je Codespace</strong><br>De ontvanger blijft op de Droplet draaien wanneer je Codespaces stopt. De Droplet heeft eigen kosten. Verwijder de server of database niet zolang je die voor de volgende oefening nodig hebt. Stem een stop van de broker af met je partner.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

1. Vul je antwoorden in en kies **File → Save All**.
2. Selecteer in **Source Control** notebook 05, `gegevens/inlevering/05_broncontrole.json` en `gegevens/inlevering/05_ontvangstcontrole.json`.
3. Controleer dat er geen wachtwoorden of geheime instellingen in de commit staan. Commit met bijvoorbeeld **Eigen en partnergegevens ontvangen en gecontroleerd** en kies **Push**.
4. Controleer op GitHub jouw branch en de opgeslagen resultaten. Bewaar je Codespaces-werkbestanden afzonderlijk wanneer je de Codespace wilt verwijderen.
5. Stop je Codespace via [github.com/codespaces](https://github.com/codespaces). Spreek af welke serverdiensten blijven draaien. Gebruik onderstaande opdracht alleen als je bewust de ontvanger wilt stoppen. Start hem later met `sudo systemctl start iot-ontvanger`.

</div>

```bash
sudo systemctl stop iot-ontvanger
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [ ] Ik kan aanwijzen waar de bron- en verzameldatabase staan.
- [ ] Mijn ontvanger ontvangt eerst eigen gegevens en daarna ook partnergegevens.
- [ ] Beide bronnen zijn volledig en inhoudelijk gecontroleerd.
- [ ] Herpublicatie levert geen extra unieke metingen op.
- [ ] Een ongeldig bericht stopt de ontvanger niet.
- [ ] Ik kan de dienst starten, stoppen, herstarten en het logboek bekijken.
- [ ] Mijn antwoorden en controlebestanden staan op mijn eigen branch.
- [ ] Mijn Codespace is gestopt en de serverafspraken met mijn partner zijn duidelijk.

</div>

## Hulp bij veelvoorkomende problemen

| Probleem | Controle en herstel |
|---|---|
| Ontvangstbestand ontbreekt | Maak het overzicht op de server en voer de juiste scp-opdracht uit in Codespaces. |
| Service active maar geen gegevens | Controleer ABONNEMENT GEREED voor iedere bron. Publiceer pas daarna opnieuw. |
| Aanmelding geweigerd | Gebruik voor ontvangst het lezerswachtwoord van de betreffende broker. Herstel met de instelhulp en herstart de ontvanger. |
| Partnercertificaat wordt geweigerd | Controleer bestand, vertrouwd bevestigde vingerafdruk, serveradres en geldigheid. Zet certificaatcontrole niet uit. |
| Eigen gegevens wel, partnergegevens niet | Controleer de partnercode, rol, brokerstatus, account lezer en topicrechten. De partner moet ook publiceren. |
| Minder records dan verwacht | Controleer of beide volledige publicaties klaar zijn. Maak daarna een nieuw overzicht. Herpubliceer zo nodig de ontbrekende bron. |
| Aantallen kloppen maar hash verschilt | Controleer broncontrolebestand, partnerkeuze, gewijzigde records en hokmetadata. Onderzoek conflicten met de docent. Wis geen database om een controle te laten slagen. |
| Ongeldig of conflict in logboek | Bekijk tellers en de gepubliceerde bron. Bestaande records zijn behouden. De oefening ondersteunt toevoegingen, geen updates van bestaande metingen. |
| Controlebestand is verouderd | Exporteer het opnieuw op de server en download opnieuw. |
| Database locked of opslagfout | Sluit handmatige schrijfprogramma's, controleer vrije schijfruimte en het logboek. De service probeert opnieuw te starten. Herpubliceer na herstel. |
| Ontvanger opnieuw verbinden | Wacht op de abonnementsbevestiging. De herstelpogingen lopen op tot maximaal dertig seconden tussen pogingen. |

## Technische bronnen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [Eclipse Paho: MQTT-client, herverbinden en handmatige ontvangstbevestiging](https://eclipse.dev/paho/files/paho.mqtt.python/html/client.html)
- [SQLite: gelijktijdig lezen en schrijven met WAL](https://www.sqlite.org/wal.html)
- [systemd: services en herstartgedrag](https://www.freedesktop.org/software/systemd/man/latest/systemd.service.html)

De ontvanger bevestigt een bericht pas nadat de opslagtransactie of afwijzing is verwerkt. De brokerbevestiging aan de publisher blijft daarvan gescheiden. De documentatie is gecontroleerd op 18 november 2026.

</div>